In [2]:
# install google OR tools
# %pip install ortools

In [3]:
# import modules needed for TSP
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import math

In [4]:
# locations of the drills in the circuit board
def create_data_model():
    data = {}
    data["locations"] = [
        (288, 149), (288, 129), (270, 133), (256, 141), (256, 157), (246, 157),
        (236, 169), (228, 169), (228, 161), (220, 169), (212, 169), (204, 169),
        (196, 169), (188, 169), (196, 161), (188, 145), (172, 145), (164, 145),
        (156, 145), (148, 145), (140, 145), (148, 169), (164, 169), (172, 169),
        (156, 169), (140, 169), (132, 169), (124, 169), (116, 161), (104, 153),
        (104, 161), (104, 169), (90, 165), (80, 157), (64, 157), (64, 165),
        (56, 169), (56, 161), (56, 153), (56, 145), (56, 137), (56, 129),
        (56, 121), (40, 121), (40, 129), (40, 137), (40, 145), (40, 153),
        (40, 161), (40, 169), (32, 169), (32, 161), (32, 153), (32, 145),
        (32, 137), (32, 129), (32, 121), (32, 113), (40, 113), (56, 113),
        (56, 105), (48, 99), (40, 99), (32, 97), (32, 89), (24, 89),
        (16, 97), (16, 109), (8, 109), (8, 97), (8, 89), (8, 81),
        (8, 73), (8, 65), (8, 57), (16, 57), (8, 49), (8, 41),
        (24, 45), (32, 41), (32, 49), (32, 57), (32, 65), (32, 73),
        (32, 81), (40, 83), (40, 73), (40, 63), (40, 51), (44, 43),
        (44, 35), (44, 27), (32, 25), (24, 25), (16, 25), (16, 17),
        (24, 17), (32, 17), (44, 11), (56, 9), (56, 17), (56, 25),
        (56, 33), (56, 41), (64, 41), (72, 41), (72, 49), (56, 49),
        (48, 51), (56, 57), (56, 65), (48, 63), (48, 73), (56, 73),
        (56, 81), (48, 83), (56, 89), (56, 97), (104, 97), (104, 105),
        (104, 113), (104, 121), (104, 129), (104, 137), (104, 145), (116, 145),
        (124, 145), (132, 145), (132, 137), (140, 137), (148, 137), (156, 137),
        (164, 137), (172, 125), (172, 117), (172, 109), (172, 101), (172, 93),
        (172, 85), (180, 85), (180, 77), (180, 69), (180, 61), (180, 53),
        (172, 53), (172, 61), (172, 69), (172, 77), (164, 81), (148, 85),
        (124, 85), (124, 93), (124, 109), (124, 125), (124, 117), (124, 101),
        (104, 89), (104, 81), (104, 73), (104, 65), (104, 49), (104, 41),
        (104, 33), (104, 25), (104, 17), (92, 9), (80, 9), (72, 9),
        (64, 21), (72, 25), (80, 25), (80, 25), (80, 41), (88, 49),
        (104, 57), (124, 69), (124, 77), (132, 81), (140, 65), (132, 61),
        (124, 61), (124, 53), (124, 45), (124, 37), (124, 29), (132, 21),
        (124, 21), (120, 9), (128, 9), (136, 9), (148, 9), (162, 9),
        (156, 25), (172, 21), (180, 21), (180, 29), (172, 29), (172, 37),
        (172, 45), (180, 45), (180, 37), (188, 41), (196, 49), (204, 57),
        (212, 65), (220, 73), (228, 69), (228, 77), (236, 77), (236, 69),
        (236, 61), (228, 61), (228, 53), (236, 53), (236, 45), (228, 45),
        (228, 37), (236, 37), (236, 29), (228, 29), (228, 21), (236, 21),
        (252, 21), (260, 29), (260, 37), (260, 45), (260, 53), (260, 61),
        (260, 69), (260, 77), (276, 77), (276, 69), (276, 61), (276, 53),
        (284, 53), (284, 61), (284, 69), (284, 77), (284, 85), (284, 93),
        (284, 101), (288, 109), (280, 109), (276, 101), (276, 93), (276, 85),
        (268, 97), (260, 109), (252, 101), (260, 93), (260, 85), (236, 85),
        (228, 85), (228, 93), (236, 93), (236, 101), (228, 101), (228, 109),
        (228, 117), (228, 125), (220, 125), (212, 117), (204, 109), (196, 101),
        (188, 93), (180, 93), (180, 101), (180, 109), (180, 117), (180, 125),
        (196, 145), (204, 145), (212, 145), (220, 145), (228, 145), (236, 145),
        (246, 141), (252, 125), (260, 129), (280, 133)
    ]
    data["depot"] = 0
    data["num_vehicles"] = 1
    return data

In [5]:
# coordinates to lengths using euclidean distance
def compute_euclidean_distance_matrix(locations):
    distances = {}
    for from_counter, from_node in enumerate(locations):
        distances[from_counter] = {}
        for to_counter, to_node in enumerate(locations):
            if from_counter == to_counter:
                distances[from_counter][to_counter] = 0
            else:
                # euclidean distance
                distances[from_counter][to_counter] = int(
                    math.hypot((from_node[0] - to_node[0]), (from_node[1] - to_node[1]))
                )
    return distances

In [6]:
# function to print the solution
def print_solution(manager, routing, solution):
    print(f"Objective : {solution.ObjectiveValue()}")
    index = routing.Start(0)
    plan_output = "Route: \n"
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += f" {manager.IndexToNode(index)} -> "
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += f" {manager.IndexToNode(index)}\n"
    plan_output += f"Objective: {route_distance}m"
    print(plan_output)

In [7]:
# main program - using First Solution(very fast, not optimal) => cheapest ARC at each node
def main():
    # create the data
    data = create_data_model()
    # create the routing index manager
    manager = pywrapcp.RoutingIndexManager(
        len(data["locations"]), data["num_vehicles"], data["depot"]
    )
    # create routing model
    routing = pywrapcp.RoutingModel(manager)
    # create the matrix with distances
    distance_matrix = compute_euclidean_distance_matrix(data["locations"])
    # function to find distance
    def distance_callback(from_index, to_index):
        # returns the distance between the two nodes
        # convert the routing variable index to distance matrix node index
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return distance_matrix[from_node][to_node]
    # registering distance call back function
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    # define distance as the cost for each arc
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # setting first solution heuristic
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    # solve the problem
    solution = routing.SolveWithParameters(search_parameters)
    # print the solution
    if solution:
        print_solution(manager, routing, solution)

# call the program
if __name__ == "__main__":
    main()


Objective : 2790
Route: 
 0 ->  1 ->  279 ->  2 ->  278 ->  277 ->  248 ->  247 ->  243 ->  242 ->  241 ->  240 ->  239 ->  238 ->  245 ->  244 ->  246 ->  249 ->  250 ->  229 ->  228 ->  231 ->  230 ->  237 ->  236 ->  235 ->  234 ->  233 ->  232 ->  227 ->  226 ->  225 ->  224 ->  223 ->  222 ->  218 ->  221 ->  220 ->  219 ->  202 ->  203 ->  204 ->  205 ->  207 ->  206 ->  211 ->  212 ->  215 ->  216 ->  217 ->  214 ->  213 ->  210 ->  209 ->  208 ->  251 ->  254 ->  255 ->  257 ->  256 ->  253 ->  252 ->  139 ->  140 ->  141 ->  142 ->  143 ->  199 ->  201 ->  200 ->  195 ->  194 ->  193 ->  191 ->  190 ->  189 ->  188 ->  187 ->  163 ->  164 ->  165 ->  166 ->  167 ->  168 ->  169 ->  171 ->  170 ->  172 ->  105 ->  106 ->  104 ->  103 ->  107 ->  109 ->  110 ->  113 ->  114 ->  116 ->  117 ->  61 ->  62 ->  63 ->  65 ->  64 ->  84 ->  85 ->  115 ->  112 ->  86 ->  83 ->  82 ->  87 ->  111 ->  108 ->  89 ->  90 ->  91 ->  102 ->  101 ->  100 ->  99 ->  98 ->  97 ->  96 ->  95 -> 

In [8]:
# main program - using Local Search Stratergy, Guided Local Search
def main():
    # create the data
    data = create_data_model()
    # create Routing Index Manager using above data
    manager = pywrapcp.RoutingIndexManager(
        len(data["locations"]), data["num_vehicles"], data["depot"]
    )
    # create Routing Model for above manager
    routing = pywrapcp.RoutingModel(manager)
    # calculate Eucledean Distance for all the locations
    distance_matrix = compute_euclidean_distance_matrix(data["locations"])
    # function to callback distance
    def distance_callback(from_index, to_index):
        # returns the distance between the nodes
        # convert the routing variable index to distance matrix node index
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return distance_matrix[from_node][to_node]
    # register "distance_callback" function as Transit Call Back Function
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    # define cost for each arc -> distance <- called by above function
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # setting guided local search
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 120
    search_parameters.log_search = True
    # solve the problem
    solution = routing.SolveWithParameters(search_parameters)
    # print the solution
    if solution:
        print_solution(manager, routing, solution)

if __name__ == "__main__":
    main()

Objective : 2597
Route: 
 0 ->  4 ->  5 ->  6 ->  7 ->  8 ->  9 ->  10 ->  11 ->  12 ->  14 ->  13 ->  23 ->  22 ->  24 ->  21 ->  25 ->  26 ->  27 ->  28 ->  29 ->  30 ->  31 ->  32 ->  33 ->  34 ->  35 ->  36 ->  37 ->  38 ->  39 ->  40 ->  41 ->  42 ->  59 ->  60 ->  58 ->  43 ->  44 ->  45 ->  46 ->  47 ->  48 ->  49 ->  50 ->  51 ->  52 ->  53 ->  54 ->  55 ->  56 ->  57 ->  67 ->  68 ->  66 ->  69 ->  70 ->  71 ->  72 ->  73 ->  74 ->  75 ->  76 ->  77 ->  78 ->  80 ->  79 ->  92 ->  93 ->  94 ->  95 ->  96 ->  97 ->  98 ->  99 ->  100 ->  101 ->  102 ->  91 ->  90 ->  89 ->  88 ->  108 ->  111 ->  87 ->  81 ->  82 ->  83 ->  86 ->  112 ->  115 ->  85 ->  84 ->  65 ->  64 ->  63 ->  62 ->  61 ->  117 ->  116 ->  114 ->  113 ->  110 ->  109 ->  107 ->  103 ->  104 ->  105 ->  106 ->  173 ->  172 ->  170 ->  171 ->  169 ->  168 ->  167 ->  166 ->  165 ->  164 ->  163 ->  162 ->  161 ->  160 ->  174 ->  159 ->  158 ->  157 ->  156 ->  118 ->  119 ->  120 ->  121 ->  122 ->  123 ->  